In [1]:
import numpy as np
import librosa
import tensorflow as tf
import os
import random

from tensorflow import keras

I0000 00:00:1773384615.219351    5972 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1773384616.364781    5972 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773384619.559932    5972 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
model = keras.models.load_model("../src/cnn_speech_model.keras")

print("Model loaded successfully")

Model loaded successfully


E0000 00:00:1773384622.740175    5972 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1773384622.740591    6031 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1773384622.772306    5972 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [3]:
max_len = 44

def extract_mfcc(file):

    audio, sr = librosa.load(file, sr=16000)

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=40
    )

    if mfcc.shape[1] < max_len:

        pad_width = max_len - mfcc.shape[1]
        mfcc = np.pad(mfcc, ((0,0),(0,pad_width)), mode='constant')

    else:
        mfcc = mfcc[:, :max_len]

    return mfcc

In [4]:
data_path = "../data"

labels = sorted([
    l for l in os.listdir(data_path)
    if os.path.isdir(os.path.join(data_path, l))
    and l != "_background_noise_"
])

print(labels)

['backward', 'bed', 'bird', 'cat', 'dog', 'down', 'eight', 'five', 'follow', 'forward', 'four', 'go', 'happy', 'house', 'learn', 'left', 'marvin', 'nine', 'no', 'off', 'on', 'one', 'right', 'seven', 'sheila', 'six', 'stop', 'three', 'tree', 'two', 'up', 'visual', 'wow', 'yes', 'zero']


In [5]:
word = "yes"

folder = f"../data/{word}"

file = random.choice(os.listdir(folder))

file_path = f"{folder}/{file}"

print("Testing file:", file_path)

Testing file: ../data/yes/ef3367d9_nohash_0.wav


In [6]:
mfcc = extract_mfcc(file_path)

mfcc = mfcc[..., np.newaxis]

mfcc = mfcc[np.newaxis, ...]

print(mfcc.shape)

(1, 40, 44, 1)


In [7]:
prediction = model.predict(mfcc)

predicted_index = np.argmax(prediction)

print("Predicted word:", labels[predicted_index])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 196ms/step
Predicted word: yes


In [8]:
for word in ["yes", "no", "up", "down"]:

    folder = f"../data/{word}"

    file = random.choice(os.listdir(folder))

    file_path = f"{folder}/{file}"

    mfcc = extract_mfcc(file_path)

    mfcc = mfcc[..., np.newaxis]
    mfcc = mfcc[np.newaxis,...]

    pred = model.predict(mfcc)

    predicted = labels[np.argmax(pred)]

    print(file_path, "→", predicted)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
../data/yes/29fb33da_nohash_2.wav → yes
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step
../data/no/5b09db89_nohash_1.wav → no
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
../data/up/23abe1c9_nohash_1.wav → up
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
../data/down/a3fc7884_nohash_0.wav → down


In [9]:
def predict_audio(file):

    mfcc = extract_mfcc(file)

    mfcc = mfcc[..., np.newaxis]

    mfcc = mfcc[np.newaxis,...]

    prediction = model.predict(mfcc)

    predicted_word = labels[np.argmax(prediction)]

    print("Predicted word:", predicted_word)

In [10]:
import sounddevice as sd
import scipy.io.wavfile as wav

def record_audio(filename="test.wav", duration=1, fs=16000):

    print("🎤 Speak now...")

    recording = sd.rec(
        int(duration * fs),
        samplerate=fs,
        channels=1
    )

    sd.wait()

    wav.write(filename, fs, recording)

    print("Recording saved:", filename)

In [11]:
record_audio("test.wav")

predict_audio("test.wav")

🎤 Speak now...
Recording saved: test.wav
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Predicted word: backward


In [12]:
import sounddevice as sd
import soundfile as sf

data, fs = sf.read("test.wav")

sd.play(data, fs)
sd.wait()